# Phase 2 — Dataset Validation

Runs a fully non-destructive integrity check against `original/`. Corrupt candidates are listed in the integrity report; no files are deleted or modified. Empty label files are valid background samples and remain untouched.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from weapon_threat_detection.artifacts import configure_logger, utc_timestamp, write_json
from weapon_threat_detection.config import load_config
from weapon_threat_detection.validation import (
    build_dataset_statistics,
    build_integrity_report,
    summarize_records,
    validate_dataset,
    write_validation_csv,
)

config = load_config(ROOT / 'configs' / 'project.yaml')
logger = configure_logger(ROOT / config['project']['logs_dir'], 'dataset_validation')
reports_dir = ROOT / config['project']['reports_dir']
records = validate_dataset(
    ROOT / config['project']['original_data_dir'],
    config['dataset']['splits'],
    len(config['dataset']['class_names']),
    config['dataset']['image_extensions'],
)
stamp = utc_timestamp()
records_path = write_validation_csv(records, reports_dir / f'validation_records_{stamp}.csv')
integrity = build_integrity_report(records, config['dataset']['splits'])
statistics = build_dataset_statistics(records, config['dataset']['class_names'])
integrity_path = write_json(reports_dir / f'integrity_report_{stamp}.json', integrity)
statistics_path = write_json(reports_dir / f'dataset_statistics_{stamp}.json', statistics)
logger.info('Validation completed: %s', summarize_records(records))
logger.info('No source files were changed. Integrity issues: %s', integrity['issue_count'])
print(f'Validation records: {records_path}')
print(f'Integrity report: {integrity_path}')
print(f'Dataset statistics: {statistics_path}')
print(integrity['status_counts'])

Validation records: /Users/mohamedtarek/weapon130/reports/validation_records_20260723T080851Z.csv
Integrity report: /Users/mohamedtarek/weapon130/reports/integrity_report_20260723T080851Z.json
Dataset statistics: /Users/mohamedtarek/weapon130/reports/dataset_statistics_20260723T080851Z.json
{'valid': 117903, 'invalid_label': 518, 'valid_background': 12342}
